# GUI 3: MCP Integration

This notebook connects MCP servers to the coding agent, giving it access to any tool exposed over the Model Context Protocol. For background on the protocol itself — FastMCP server construction, the `Client` API, and available transport types — see the [MCP notebook](/notebooks/tooling/mcp.html). Here we focus entirely on the agent side: how MCP tools are discovered at startup, adapted into the `ToolRegistry`, and surfaced in the UI.

Three new objects carry the integration: `MCPServerConfig` declares how to reach a server, `MCPToolAdapter` wraps a single discovered tool as a first-class `Tool`, and `MCPManager` owns the connection lifecycle for the whole app. On the UI side, `app.py` opens connections after layout via `page.run_task`, routes `/mcp` through `handle_command`, and renders MCP tool cards in purple to distinguish them from the blue builtin cards.

## `mcp_bridge.py`

All three MCP classes live in `src/notebooks/agent/tools/mcp_bridge.py` and are re-exported from both `notebooks.agent.tools` and `notebooks.agent`.

**`MCPServerConfig`.** A dataclass with two mutually exclusive modes. Set `command` + `args` (and optionally `env`) for a local subprocess over stdio. Set `url` (and optionally `headers`) for a remote HTTP server. The `name` field serves as the tool-name prefix — every tool from the server is namespaced as `{name}__{mcp_tool_name}`. The `transport_kind` property returns `"stdio"` or `"http"` accordingly, and `__post_init__` raises `ValueError` if neither or both modes are specified.

**`MCPToolAdapter`.** A `Tool` subclass with `kind = ToolKind.NETWORK`. The constructor takes the server name, the original MCP tool name, its description, its JSON Schema `inputSchema`, and the open `fastmcp.Client` to call through. Rather than generating a Pydantic model from the schema, it stores the raw dict directly as `self.schema = {"parameters": input_schema}`, which the agent's OpenAI schema serializer picks up unchanged. The tool's visible name is `{server_name}__{mcp_tool_name}`; the description is prefixed `[{server_name}]` so the agent always knows which server a tool comes from. `execute()` calls `client.call_tool(mcp_tool_name, invocation.params)` and unpacks `TextContent` items from the result.

**`MCPManager`.** Owns open `fastmcp.Client` instances for the duration of the application session. `connect_all(registry, skip_errors=True)` iterates the configured servers, opens each client via `client.__aenter__()`, calls `list_tools()`, wraps each tool as an `MCPToolAdapter`, and registers it. `close()` calls `client.__aexit__()` on every open client. `status()` returns a dict of `{server_name: {"connected": bool, "tools": [str]}}` — the raw output of `/mcp`.

We verify the adapter end-to-end using an in-process FastMCP server. `fastmcp.Client` accepts a `FastMCP` instance directly as its transport argument, so no subprocess or network socket is needed:

**In-process server.** Defining a minimal math server:

In [ ]:
import fastmcp

math_server = fastmcp.FastMCP("math")

@math_server.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@math_server.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

**Registry setup.** Creating the default registry and the adapter infrastructure:

In [ ]:
from notebooks.agent.tools.mcp_bridge import MCPManager, MCPServerConfig, MCPToolAdapter
from notebooks.agent.tools.registry import create_default_registry
from notebooks.agent.config import Config

config = Config()
registry = create_default_registry(config)
print(f"Builtin tools before MCP: {len(registry.get_tools())}")

**Adapter registration.** Opening the in-process client, listing tools, and registering each as an `MCPToolAdapter`:

In [ ]:
from fastmcp import Client
from pathlib import Path

client = Client(math_server)
await client.__aenter__()

mcp_tools = await client.list_tools()
print(f"Discovered tools: {[t.name for t in mcp_tools]}")

for mcp_tool in mcp_tools:
    adapter = MCPToolAdapter(
        server_name="math",
        mcp_tool_name=mcp_tool.name,
        description=mcp_tool.description or "",
        input_schema=mcp_tool.inputSchema,
        client=client,
        config=config,
    )
    registry.register(adapter)
    print(f"Registered: {adapter.name}  kind={adapter.kind.value}")

**End-to-end invocation.** Calling through the registry to verify the full adapter path:

In [ ]:
result = await registry.invoke("math__add", {"a": 3, "b": 4}, Path.cwd())
print(f"math__add(3, 4) \u2192 {result.output}")

result2 = await registry.invoke("math__multiply", {"a": 6, "b": 7}, Path.cwd())
print(f"math__multiply(6, 7) \u2192 {result2.output}")

await client.__aexit__(None, None, None)

**`MCPServerConfig` transport kinds.** The two production patterns — stdio subprocess and HTTP remote server:

In [ ]:
# stdio server (local subprocess)
stdio_cfg = MCPServerConfig(
    name="mytools",
    command="python",
    args=["path/to/server.py"],
)
print(f"stdio  transport: {stdio_cfg.transport_kind}")

# HTTP server (remote)
http_cfg = MCPServerConfig(
    name="context7",
    url="https://mcp.context7.com/mcp",
)
print(f"http   transport: {http_cfg.transport_kind}")

# Pass to Config — mcp_servers is list[Any] to avoid circular imports
cfg = Config(mcp_servers=[stdio_cfg, http_cfg])
print(f"Configured servers: {[s.name for s in cfg.mcp_servers]}")

**`MCPManager.status()`.** The manager tracks connected clients and registered tool names internally:

In [ ]:
# Empty manager — no servers configured
manager = MCPManager([])
print("empty:", manager.status())

# After connect_all the status dict reflects live state:
# { "math": {"connected": True, "tools": ["math__add", "math__multiply"]} }

## `/mcp` Command

The previous notebook added `handle_command(raw, agent, session)` in `ui/commands.py`. This notebook adds one new command — `/mcp` — and an optional `mcp_manager` keyword argument to `handle_command`. The signature is now:

```python
def handle_command(
    raw: str,
    agent: Agent,
    session: Session,
    mcp_manager: MCPManager | None = None,
) -> CommandResult:
```

When `mcp_manager` is `None` and `/mcp` is typed, the command returns `"No MCP manager configured."` When the manager is provided, it calls `manager.status()` and formats the result as a server table — one entry per server showing connection state and the list of registered tools. All other commands are unaffected by the new parameter.

**`/mcp` demo.** Using a fake manager to show the formatted output:

In [ ]:
from notebooks.agent.ui.commands import handle_command, CommandResult
from notebooks.agent.agent import Agent
from notebooks.agent.config import Config

config = Config()
agent = Agent(config)
session = agent.session


class FakeManager:
    def status(self):
        return {
            "math":     {"connected": True,  "tools": ["math__add", "math__multiply"]},
            "context7": {"connected": False, "tools": []},
        }


result = handle_command("/mcp", agent, session, mcp_manager=FakeManager())
print(result.message)

**`/mcp` without a manager.** Confirms the graceful fallback:

In [ ]:
result = handle_command("/mcp", agent, session)
print(result.message)

**`/help`.** The updated help text includes the new command:

In [ ]:
result = handle_command("/help", agent, session)
print(result.message)

## UI Integration

Four changes to `app.py` wire the `MCPManager` into the Flet application lifecycle.

**Construction.** `AgentApp.__init__` creates the manager immediately after the agent:

```python
self._mcp_manager = MCPManager(self.config.mcp_servers)
```

The manager is constructed eagerly but does not open any connections yet — connections happen after the layout is built so that errors can be reported as system bubbles in the feed.

**Async connect.** After `_build_layout()` completes, `__init__` schedules the connection task:

```python
page.run_task(self._connect_mcp)
```

`_connect_mcp` calls `connect_all(registry, skip_errors=True)`. For each server it adds a system bubble: `"MCP: connected to '{name}' — N tool(s) registered."` on success, or `"MCP: failed to connect to '{name}' (check logs)."` on failure. The `skip_errors=True` default means a downed server never prevents the app from starting.

**Shutdown.** `page.on_disconnect` is assigned to `_on_disconnect`, which calls `await self._mcp_manager.close()`. This closes every open `fastmcp.Client` cleanly when the window or browser tab is closed.

**Command routing.** The `/mcp` command is forwarded automatically because `_on_send` passes the manager to `handle_command`:

```python
result = handle_command(text, self.agent, self.agent.session, self._mcp_manager)
```

**Tool card color.** `make_tool_call_card` in `components.py` detects MCP tools by the double-underscore namespace separator in the name:

```python
name_color = ft.Colors.PURPLE_200 if "__" in tc.name else ft.Colors.BLUE_200
```

Builtin tool cards remain blue; MCP tool cards render in purple, giving an immediate visual signal of which calls cross a server boundary.

The updated architecture diagram:

```
AgentApp (page controller)
├── Agent (agentic loop)
│   └── Session → LLMClient + ToolRegistry (11 + N MCP tools)
├── ApprovalManager (approval decisions)
├── MCPManager  ← NEW: owns open MCP client connections
│   ├── connect_all(registry)  called via page.run_task on startup
│   └── close()                called on page.on_disconnect
├── commands.py  (slash command parser, /mcp added)
└── Flet Page
    ├── StatusBar (model · turn · tokens)
    ├── ListView (message feed)
    │   ├── MessageBubble (user / assistant / system)
    │   └── ToolCallCard (blue = builtin, purple = MCP)  ← NEW
    └── InputBar (TextField + Send button)
```

**Package exports.** The three MCP classes are part of the top-level `notebooks.agent` public API:

In [ ]:
import notebooks.agent as agent_pkg

mcp_exports = [s for s in agent_pkg.__all__ if "MCP" in s]
print("MCP exports:", mcp_exports)

## Running the App

The launch commands are unchanged from the previous notebook. As a desktop app:

```{.bash filename="$ (local)"}
OPENROUTER_API_KEY=sk-... uv run flet run src/notebooks/agent/ui/app.py
```

As a web app served on `localhost:8550`:

```{.bash filename="$ (local)"}
OPENROUTER_API_KEY=sk-... uv run flet run --web --port 8550 src/notebooks/agent/ui/app.py
```

To add MCP servers, override `_build_config` in `app.py`:

```python
from notebooks.agent.tools.mcp_bridge import MCPServerConfig
from notebooks.agent.config import Config, ApprovalPolicy
from pathlib import Path

def _build_config() -> Config:
    return Config(
        cwd=Path.cwd(),
        approval=ApprovalPolicy.ON_REQUEST,
        max_turns=50,
        mcp_servers=[
            MCPServerConfig(
                name="context7",
                url="https://mcp.context7.com/mcp",
            ),
        ],
    )
```

On the next launch, the app will attempt to connect to `context7` immediately after the layout renders. A system bubble in the feed will confirm success or report failure. Type `/mcp` at any time to inspect the current connection state and the full list of registered MCP tools.

:::{.callout-note}
`MCPManager.connect_all` is called with `skip_errors=True` by default.
A server that is offline at startup does not prevent the app from
launching — the failure is reported as a system bubble in the feed.
The `/mcp` command can be used at any time to check which servers are
connected and which tools they have registered.

:::

---

■

## Conclusion

We have built a complete async coding agent from a streaming LLM client through tool execution, session management, hardening, configuration loading, hooks, sub-agents, and a Flet desktop interface with MCP integration — each layer a self-contained module that can be used or replaced independently. The series mirrors the architecture of Claude Code style agents: a thin protocol layer (`LLMClient`), a composable tool system, a turn-based agentic loop, and a stack of cross-cutting concerns (context, approval, loop detection, hooks, config) that plug in without touching the core.

← [GUI 2: Tools & Persistence](/notebooks/apps/cda/08-ui2.html)